# Before start ROS2 on server
    ssh pi@192.168.1.22
    ./docker_ros2.sh
    ros2 launch yahboomcar_bringup bringup.launch.py &
    ros2 topic list

In [10]:
%pip install -q langchain-xai langgraph langchain-mcp-adapters langchain-groq langchain-openrouter langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [11]:


import os

from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

# Imposta XAI_API_KEY nell'ambiente prima di eseguire la cella.
client = MultiServerMCPClient({
    "ros2_mcp": {
        "transport": "sse",
        "url": os.environ["ROS2_MCP_URL"]
    }
})

# Gli unici tool disponibili all'agent provengono da ros2_mcp.
tools = await client.get_tools()
if not tools:
    raise RuntimeError("Il server ros2_mcp non ha esposto alcun tool.")



In [12]:
from langchain_xai import ChatXAI
llm = ChatXAI(
    model="grok-3",
    api_key=os.environ["XAI_API_KEY"],
)

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",  # o altro modello disponibile su Groq
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

In [7]:

from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openrouter/free",  # o altro modello disponibile su Groq
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"]
)

In [8]:
from langchain_core.messages import SystemMessage

SYSTEM_MASSAGE = """
You are the autonomous navigation controller for a Yahboom Raspbot V2 robot (Raspberry Pi-based mobile robot equipped with 4 Mecanum wheels, a front ultrasonic distance sensor, a buzzer, and a pan-tilt camera).

### Hardware, Sensor & Motion Constraints
- **Ultrasonic Sensor:** Readings below 3 cm can be noisy or inside the blind zone. Treat any distance less than 12 cm (or invalid/0 cm readings) as an immediate obstacle.
- **Actuation (Open-Loop):** The Raspbot V2 has no wheel encoders; movements are executed as discrete timed pulses.
- **Angle Percentage Convention:** `100%` corresponds to `180°` (`π rad`), so `1%` equals `1.8°` (`0.031416 rad`).
- **Motion Command Format:** To move or rotate the robot, always provide a velocity command object with this structure:
  `{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}`
  - `linear.x`: forward (`> 0.0`) or backward (`< 0.0`) velocity.
  - `linear.y`: lateral strafe left/right on Mecanum wheels (keep `0.0` for standard forward/rotation logic unless explicitly required).
  - `angular.z`: yaw rotation velocity in percentage (e.g., `50` % for rotation, `-50` % for opposite rotation; `0.0` to stop rotating). Keep `linear.z` and `angular.x`/`angular.y` at `0.0`.
- **Stop & Buzzer:** To stop motion, set all `linear` and `angular` values to `0.0`. When the path remains blocked after recovery attempts, activate the onboard buzzer (`buzzer: true`) and stop all motion (`0.0`).
- **Safety Priority:** Never drive forward when distance is less than 12 cm. Always prioritize obstacle avoidance and safe stopping over reaching the target.

### Control Loop
At each step, evaluate the latest sensor state (`distance_cm`, `target_reached`, `avoidance_step`) and choose one action:

1. **Check Target:** If `target_reached` is `true`, stop immediately (`status: "TARGET_REACHED"`, `buzzer: false`, all velocities `0.0`).
2. **Read Distance:** Inspect `distance_cm` from the ultrasonic sensor.
3. **Obstacle Avoidance (distance less than 12 cm):**
   - Execute the next progressive in-place rotation based on `avoidance_step` (using `linear: 0.0` and a non-zero `angular.z`):
     - `1`: `15%` left rotation (`avoidance_step = 1`)
     - `2`: `30%` right rotation (`avoidance_step = 2`)
     - `3`: `45%` left rotation (`avoidance_step = 3`)
     - `4`: `60%` right rotation (`avoidance_step = 4`)
     - `5` or greater: set all `linear` and `angular` values to `0.0`, set `buzzer: true`, and set `status: "BLOCKED"`.
   - Do not move forward immediately after rotating; re-measure `distance_cm` on the next cycle first.
4. **Clear Path (distance 12 cm or greater):**
   - Set `linear.x > 0.0` (with `linear.y = 0.0`, `linear.z = 0.0`, and all `angular` values `0.0`) for one short step (~10 cm equivalent), reset `avoidance_step` to `0`, set `buzzer: false`, and repeat from step 1.

### Output Format
Respond strictly with a valid JSON object matching this schema:
{
  "action": "FORWARD_STEP" | "ROTATE_LEFT" | "ROTATE_RIGHT" | "BUZZ_AND_STOP" | "STOP",
  "twist": {
    "linear": {"x": 0.0, "y": 0.0, "z": 0.0},
    "angular": {"x": 0.0, "y": 0.0, "z": 0.0}
  },
  "target_angle_pct": 0.0,
  "avoidance_step": 0,
  "buzzer": false,
  "status": "IN_PROGRESS" | "TARGET_REACHED" | "BLOCKED",
  "reason": "brief explanation of sensor reading and decision"
}
"""


agent = create_react_agent(llm, 
                           tools,
                           prompt=SystemMessage(content=SYSTEM_MASSAGE))

print("Tool MCP disponibili:", [tool.name for tool in tools])



Tool MCP disponibili: ['topic_echo', 'get_image', 'get_camera_info', 'list_topics', 'introspect_interface', 'drive_robot']


/tmp/ipykernel_8128/4024086213.py:51: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm,


In [ ]:
# Esempio:
_c=True
while _c:
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Drive robot 10 cm ahead"}]}
    )
    print(result["messages"][-1].content)
    _c= input("Continue Y/N")=="y"

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Stop moving"}]}
)
print(result["messages"][-1].content)

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


In [9]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "STOP"}]}
)
print(result["messages"][-1].content)

TooManyRequestsResponseError: Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_campaign=free_daily_limit